<a href="https://colab.research.google.com/github/Titantus/Truth-Zero-C/blob/main/%E2%9A%99%EF%B8%8FT'Z0C_ENGINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# @title ⚙️ T'Z0C CORE ENGINE — Falsifiable Sims Dashboard
# ============================================================

# Dependencies (quiet install)
import subprocess, sys, os
def _pip(pkg):
    try:
        __import__(pkg.split()[0])
    except:
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkg.split())

for pkg in ["pint", "jsonschema", "ipywidgets", "scipy"]:
    _pip(pkg)

# Imports
import json, math, uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Callable, Dict, Any, Sequence, Optional
from datetime import datetime, UTC
from jsonschema import validate, ValidationError
from IPython.display import display, clear_output
import ipywidgets as widgets
import pint

# ============================================================
# CONFIG (Updated for modular files)
# ============================================================
CORE_JSON_UNIVERSAL_INVARIANTS_PATH = Path("/content/⚙️ CORE.json") # Universal invariants
REGISTRY_ELEMENTS_PATH = Path("/content/⚙️ REGISTRY_ELEMENTS.json") # Elements and Molecules
MODULES_SIM_PATH = Path("/content/⚙️ MODULES_SIM.json") # Simulation Modules
INTERFACE_CONFIG_PATH = Path("/content/⚙️ INTERFACE_CONFIG.json") # Interface config (empty for now)

# New Paths for modular Database
DATABASE_SUMMARY_PATH = Path("/content/⚙️ DATABASE_SUMMARY.json")
DATABASE_ARCHIVE_PATH = Path("/content/⚙️ DATABASE_ARCHIVE.json")
DATABASE_EXCEPTIONS_PATH = Path("/content/⚙️ DATABASE_EXCEPTIONS.json")

OUTPUT_FOLDER = Path("tz0c_outputs")
REGISTRY_LOG = Path("tz0c_registry_load.log")
OUTPUT_FOLDER.mkdir(exist_ok=True)

# ============================================================
# DARK THEME
# ============================================================
plt.style.use("dark_background")
display(widgets.HTML("""
<style>
.widget-label { color:#ddd !important; }
.widget-dropdown, .widget-select, .widget-button {
    background:#222 !important; color:#eee !important; border-color:#444 !important;
}
.jp-OutputArea-output pre { color:#ddd; background:#111; }
</style>
"""))

# ============================================================
# REGISTRY SCHEMA (RELAXED FOR REAL DATA INGEST)
# ============================================================
REGISTRY_SCHEMA = {
    "type": "object",
    "patternProperties": {
        "^[A-Za-z0-9]+$": {
            "type": "object",
            "properties": {
                "symbol": {"type": "string"},
                "atomic_number": {"type": ["number", "null"]},
                "torque_density_alpha": {"type": ["number", "null"]},
                "invariant_angle": {"type": ["number", "null"]},
                "eta_peak": {"type": ["number", "null"]},
                "janibekov_limit": {"type": ["number", "null"]},
                "name": {"type": "string"},
                "atomic_weight": {"type": ["number", "null"]},
                "shake_hz": {"type": ["number", "null"]},
                "crystal_structure": {"type": "string"},
                "geometry_type": {"type": "string"},
                "perfect_angle": {"type": ["number", "null"]},
                "kappa_min": {"type": ["number", "null"]},
                "flip_risk": {"type": ["number", "null"]},
                "color": {"type": "string"},
                "bond_angle": {"type": ["number", "null"]},
                "gear_assignment": {"type": "string"},
                "notes": {"type": "string"}
            },
            "required": ["symbol"]  # only symbol is hard-required now
        }
    },
    "additionalProperties": False
}

# ============================================================
# QUIET REGISTRY LOADER (Updated for modular files)
# ============================================================
def load_registry_quiet():
    """Load registry quietly; write warnings to log instead of UI.
    Now loads from REGISTRY_ELEMENTS_PATH, MODULES_SIM_PATH, and CORE_JSON_UNIVERSAL_INVARIANTS_PATH."""
    elements_and_molecules_parsed = {}
    modules_parsed_filtered = []
    universal_invariants_data = {}

    try:
        # Load elements and molecules from REGISTRY_ELEMENTS.json
        if REGISTRY_ELEMENTS_PATH.exists():
            with open(REGISTRY_ELEMENTS_PATH, "r") as f:
                registry_data = json.load(f)
                elements_and_molecules_parsed.update(registry_data.get("elements", {}))
                elements_and_molecules_parsed.update(registry_data.get("molecules", {}))
        else:
            REGISTRY_LOG.write_text(f"Warning: {REGISTRY_ELEMENTS_PATH} not found.\n")

        # Load modules from MODULES_SIM.json
        if MODULES_SIM_PATH.exists():
            with open(MODULES_SIM_PATH, "r") as f:
                raw_modules = json.load(f)
                for mod in raw_modules:
                    if isinstance(mod, dict) and "simulation_name" in mod:
                        modules_parsed_filtered.append(mod)
                    else:
                        REGISTRY_LOG.write_text(f"Warning: Skipping malformed module (missing 'simulation_name'): {mod}\n")
        else:
            REGISTRY_LOG.write_text(f"Warning: {MODULES_SIM_PATH} not found.\n")

        # Load universal invariants from CORE_JSON_UNIVERSAL_INVARIANTS_PATH
        if CORE_JSON_UNIVERSAL_INVARIANTS_PATH.exists():
            with open(CORE_JSON_UNIVERSAL_INVARIANTS_PATH, "r") as f:
                core_data = json.load(f)
                universal_invariants_data = core_data.get("universal_invariants", {})
        else:
            REGISTRY_LOG.write_text(f"Warning: {CORE_JSON_UNIVERSAL_INVARIANTS_PATH} not found.\n")

        globals()["master_modules"] = modules_parsed_filtered
        globals()["universal_invariants"] = universal_invariants_data # Store universal invariants globally

        try:
            validate(elements_and_molecules_parsed, REGISTRY_SCHEMA)
        except ValidationError as e:
            REGISTRY_LOG.write_text(f"Validation warning for combined registry: {e.message}\n")

        return elements_and_molecules_parsed
    except Exception as e:
        REGISTRY_LOG.write_text(f"Critical load error: {e}\n")
        display(widgets.HTML(
            f"<b style='color:#ff6666'>Registry load error:</b> {e} — see tz0c_registry_load.log"
        ))
        return {}

# Re-run the registry load with the corrected function
registry = load_registry_quiet()

# ============================================================
# UI WIDGETS (RE-INITIALIZED FOR CORRECT DATA)
# ============================================================
# Re-initialize element_options since registry has been correctly populated
# ------------------------------------------------------------
# ELEMENT DROPDOWN — ATOMIC ORDER + MOLECULE GROUPING
# ------------------------------------------------------------
atomic_elements = []
molecules = []

for sym, el in registry.items():
    Z = el.get("atomic_number", None)
    # Classify as atomic if atomic_number is a positive integer/float
    if isinstance(Z, (int, float)) and Z > 0:
        atomic_elements.append((Z, sym))
    else:
        molecules.append(sym)

atomic_elements_sorted = [sym for Z, sym in sorted(atomic_elements, key=lambda x: x[0])]
molecules_sorted = sorted(molecules)

element_options = (
    [('— Atomic Elements —', None)] +
    [(sym, sym) for sym in atomic_elements_sorted] + # Corrected to be (label, value) tuples
    [('— Molecules —', None)] +
    [(sym, sym) for sym in molecules_sorted]           # Corrected to be (label, value) tuples
)

element_dropdown = widgets.Dropdown(
    options=element_options,
    value=element_options[1][1] if len(element_options) > 1 and element_options[1][1] is not None else None, # Default to first element value if available
    description='Element:',
    disabled=False,
)

# Re-initialize grouped_module_options since master_modules has been correctly populated
# ------------------------------------------------------------
# MODULE DROPDOWN — GROUPED + ICONS + SHORT NAMES
# ------------------------------------------------------------
# Define module categories with actual module names from the JSON
MODULE_GROUPS = {
    "Plotting 📈": [
        "Final Systems Handshake",
        "Dual Sweep Visualizer",
        "Element Angle Deviation Heatmap",
        "Shake Frequency vs Invariant Angle Scatter",
        "Priority Audit: Efficiency vs Angular Deviation"
    ],
    "Raw Data 📊": [
        "C-Ladder Scaling Analysis",
        "Torque Density vs Atomic Number Trend",
        "Master Periodic Table Generator"
    ],
    "Standalone ⚙️": [
        "Resonance Collapse Sequence",
        "Wave-Lock Interference Potential",
        "Bounce Gap Stability Monte-Carlo",
        "Falsifiability Stress Test (Lattice Integrity)",
        "Diamond Phonon Siphon Simulator"
    ]
}

grouped_module_options = []
master_mods = globals().get("master_modules", []) # Ensure master_mods is updated from load_registry_quiet

for group_name, short_names in MODULE_GROUPS.items():
    grouped_module_options.append((f"— {group_name} —", None))
    for short in short_names:
        # Safely check for 'simulation_name' before accessing
        match = next((m for m in master_mods if isinstance(m, dict) and "simulation_name" in m and m["simulation_name"] == short), None)
        if match:
            grouped_module_options.append((f"{short}", match["simulation_name"]))

# Identify uncategorized modules (if any remain)
all_categorized_names = set()
for short_names_list in MODULE_GROUPS.values():
    all_categorized_names.update(short_names_list)

# Safely filter master_mods for those with 'simulation_name' before processing uncategorized
valid_master_mods = [m for m in master_mods if isinstance(m, dict) and "simulation_name" in m]

uncategorized = [
    m["simulation_name"] for m in valid_master_mods
    if m["simulation_name"] not in all_categorized_names
]

if uncategorized:
    grouped_module_options.append(('— Other Modules —', None))
    for name in uncategorized:
        grouped_module_options.append((name, name))

module_dropdown = widgets.Dropdown(
    options=grouped_module_options,
    value=grouped_module_options[1][1] if len(grouped_module_options) > 1 and grouped_module_options[1][1] is not None else None, # Default to first module
    description='Module:',
    disabled=False,
)

# Initialize other widgets that were assumed to exist
execute_button = widgets.Button(description="Execute")
history_button = widgets.Button(description="History")
progress_label = widgets.Label("Idle")
run_output = widgets.Output()

print("Notebook code review and corrections applied. Registry and UI dropdowns re-initialized.")

# ============================================================
# DATABASE HELPERS (Updated for modular database)
# ============================================================
def _init_db():
    if not DATABASE_SUMMARY_PATH.exists():
        summary_data = {
            "metadata": {"description": "Summary of T'Z0C simulation runs"},
            "stats": {"total_runs": 0, "mean_peak": None, "efficiency_avg": None},
            "top_performers": [],
            "recent_runs": []
        }
        DATABASE_SUMMARY_PATH.write_text(json.dumps(summary_data, indent=2))
    # Ensure archive and exceptions files exist
    if not DATABASE_ARCHIVE_PATH.exists():
        DATABASE_ARCHIVE_PATH.write_text(json.dumps({"archive_records": []}, indent=2))
    if not DATABASE_EXCEPTIONS_PATH.exists():
        DATABASE_EXCEPTIONS_PATH.write_text(json.dumps({"exceptions": []}, indent=2))

def save_run(record):
    _init_db()

    # Load summary data
    summary_data = json.loads(DATABASE_SUMMARY_PATH.read_text())

    # Update total runs
    summary_data["stats"]["total_runs"] = summary_data["stats"].get("total_runs", 0) + 1

    # Add to recent runs (keep only last 10 as per instruction)
    summary_data["recent_runs"].append(record)
    summary_data["recent_runs"] = summary_data["recent_runs"][-10:] # Keep only the last 10

    # Save updated summary data
    tmp_summary = DATABASE_SUMMARY_PATH.with_suffix(".tmp")
    tmp_summary.write_text(json.dumps(summary_data, indent=2))
    os.replace(tmp_summary, DATABASE_SUMMARY_PATH)


# ============================================================
# CANONICAL KERNEL + SWEEP (LIGHTWEIGHT)
# ============================================================
ureg = pint.UnitRegistry()
Q_ = ureg.Quantity

def g_theta(theta):
    return math.exp(-0.5 * ((theta - 109.47122) / 0.5)**2)

def A_alpha(alpha):
    return alpha / (1 + abs(alpha))

def h_freq(f, f0, sigma):
    return math.exp(-0.5 * ((f - f0) / sigma)**2)

def canonical_kernel(theta, r, f, alpha, kappa, f0, sigma):
    theta = float(Q_(theta, "degree").magnitude)
    r = Q_(r, "m")
    f = Q_(f, "Hz")
    f0 = Q_(f0, "Hz")
    sigma = Q_(sigma, "Hz")
    alpha = float(alpha)
    kappa = Q_(kappa, "1/m")

    base = Q_(1.0, "newton*meter/meter**3")
    damping = math.exp(-float((kappa * r).magnitude))
    return base * A_alpha(alpha) * g_theta(theta) * damping * h_freq(f.magnitude, f0.magnitude, sigma.magnitude)

@dataclass
class SweepResult:
    theta_deg: float
    r_m: float
    f_hz: float
    alpha: float
    kappa_1_per_m: float
    K_value: float

class ParameterSweep:
    def __init__(self, kernel=canonical_kernel):
        self.kernel = kernel

    def run(self, thetas, rs, fs, alphas, kappas, f0=1.62e14, sigma=1e12):
        rows = []
        for th in thetas:
            for r in rs:
                for f in fs:
                    for a in alphas:
                        for k in kappas:
                            K = self.kernel(th, r, f, a, k, f0, sigma)
                            rows.append(SweepResult(th, r, f, a, k, float(K.magnitude)))
        return pd.DataFrame([asdict(r) for r in rows])

# ============================================================
# MODULE EXECUTION (SAFE) - UPDATED
# ============================================================
def run_module(module_dict, symbol):
    code = module_dict.get("python_code", "")
    local = {}

    try:
        exec(code, {}, local)
    except Exception as e:
        return None, f"Module exec error: {e}"

    func = next((v for v in local.values() if callable(v)), None)
    if func is None:
        return None, "No callable found in module."

    import inspect
    sig = inspect.signature(func)
    params = list(sig.parameters.keys())

    kwargs = {}
    args = []

    if len(params) >= 1:
        args.append(symbol)
    if "registry" in params:
        kwargs["registry"] = registry
    if "full_registry" in params:
        kwargs["full_registry"] = registry
    if "save_csv" in params:
        kwargs["save_csv"] = False

    try:
        out = func(*args, **kwargs)

        # Normalize output (Accept None as a 'Success without data')
        if isinstance(out, pd.DataFrame):
            return out, None
        if isinstance(out, dict) and "dataframe" in out:
            return pd.DataFrame(out["dataframe"]), None

        # NEW: If it returns nothing, but didn't crash, we return a success flag
        return "SUCCESS_NO_DATA", None

    except Exception as e:
        return None, f"Module call error: {e}"

# ============================================================
# CALLBACKS - UPDATED
# ============================================================
def on_execute(_):
    run_output.clear_output()
    sym = element_dropdown.value
    mod_name = module_dropdown.value
    mod = next((m for m in globals().get("master_modules", []) if m["simulation_name"] == mod_name), None)

    progress_label.value = f"Running {mod_name} on {sym}…"
    run_id = str(uuid.uuid4())
    timestamp = datetime.now(UTC).isoformat()

    with run_output:
        df = None # Initialize df here
        if mod:
            df, err = run_module(mod, sym)

            if err:
                print("Module error:", err)
                print("Falling back to default sweep.")
                df = default_sweep(sym)
            elif df == "SUCCESS_NO_DATA":
                print(f"✅ {mod_name} executed successfully.")
                # We skip the fallback because the module handled its own plotting
                df = None # Ensure df is None for no-data modules
        else:
            # If no module found, always run default sweep
            df = default_sweep(sym)

        # Save & Plot only if we have a dataframe (fallback or returned)
        if df is not None and not isinstance(df, str):
            outname = f"{run_id}_{sym}_{mod_name.replace(' ','_')}.csv"
            df.to_csv(OUTPUT_FOLDER / outname, index=False)
            save_run({"id": run_id, "timestamp": timestamp, "element": sym, "module": mod_name, "csv": outname})

            if "theta_deg" in df and "K_value" in df:
                plt.figure(figsize=(7,4))
                plt.plot(df["theta_deg"], df["K_value"], marker="o")
                plt.grid(alpha=0.2)
                plt.title(f"{mod_name} — {sym}")
                plt.show()
            else:
                display(df.head())
        else:
            # For modules that save their own data internally or return no DataFrame
            save_run({"id": run_id, "timestamp": timestamp, "element": sym, "module": mod_name, "csv": "Internal_Module_Save"})

        progress_label.value = f"Done: {mod_name} on {sym}"

def on_history(_):
    run_output.clear_output()
    with run_output:
        if not DATABASE_SUMMARY_PATH.exists():
            print("No history yet (Summary DB not found).")
            return
        summary_data = json.loads(DATABASE_SUMMARY_PATH.read_text())
        runs_data = summary_data.get("recent_runs", [])
        if runs_data:
            df = pd.DataFrame(runs_data)
            display(df.tail(10))
        else:
            print("No recent runs recorded in summary database.")

execute_button.on_click(on_execute)
history_button.on_click(on_history)

# ============================================================
# DEFAULT SWEEP (Updated to use universal_invariants)
# ============================================================
def default_sweep(symbol):
    el = registry.get(symbol, {})
    # Access universal_invariants from globals
    universal_invariants_access = globals().get("universal_invariants", {})

    th0 = el.get("invariant_angle", universal_invariants_access.get("tetrahedral_angle", 109.47122))

    default_f0 = universal_invariants_access.get("default_f0_hz", 1.62e14)
    default_sigma = universal_invariants_access.get("default_sigma_hz", 1e12)

    sweep = ParameterSweep()
    return sweep.run(
        [th0-2, th0-0.5, th0, th0+0.5, th0+2],
        [1e-10, 1e-9],
        [default_f0 - 0.12e14, default_f0, default_f0 + 0.18e14],
        [max(0.01, el.get("torque_density_alpha", 0.1)), max(1.0, el.get("torque_density_alpha", 1.0))],
        [1e9, 1e10],
        f0=default_f0,
        sigma=default_sigma
    )

# ============================================================
# DASHBOARD LAYOUT
# ============================================================
controls = widgets.HBox([element_dropdown, module_dropdown, execute_button, history_button, progress_label])
dashboard = widgets.VBox([controls, run_output])
display(dashboard)

print("Dashboard ready.")